In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

### 사칙 연산 툴

In [2]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """
    두 숫자를 더해 그 합을 반환합니다.

    Args:
        a: 더할 첫 번째 숫자
        b: 더할 두 번째 숫자
    """
    return a + b

@tool
def subtract(a: int, b: int) -> int:
    """
    첫 번째 숫자에서 두 번째 숫자를 뺍니다.

    Args:
        a: 빼기 연산의 대상이 되는 수 (피감수)
        b: 뺄 숫자 (감수)
    """
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """
    두 숫자를 곱해 그 결과를 반환합니다.

    Args:
        a: 곱할 첫 번째 숫자
        b: 곱할 두 번째 숫자
    """
    return a * b

@tool
def divide(a: int, b: int) -> float:
    """
    첫 번째 숫자를 두 번째 숫자로 나눕니다. 
    0으로 나누려고 하면 에러 메시지를 반환합니다.

    Args:
        a: 나뉘는 수 (피제수)
        b: 나누는 수 (제수)
    """
    if b == 0:
        return "Error: 0으로 나눌 수 없습니다."
    return a / b

# 툴 리스트
tools = [add, subtract, multiply, divide]

In [3]:
print(f"더하기 테스트: {add.invoke({'a': 10, 'b': 5})}")
print(f"나누기 테스트: {divide.invoke({'a': 10, 'b': 2})}")

더하기 테스트: 15
나누기 테스트: 5.0


In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools
)

In [8]:
# 도구 호출 하지 않을 수도 있음
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [9]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='b27982e2-ff4d-495d-a1c3-0371226c0928'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 789, "a": 456}'}, '__gemini_function_call_thought_signatures__': {'e8760268-66b2-4295-9413-edd38469bfd9': 'EjQKMgEMOdbHhPWb68W6/ga6zwC7FT0eAF8zLg3gQe7ntikG8AQJsTJOfC5Z0Z4laGnSfKfh'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0811-c6a3-7752-a7fa-ef8a9b8a61a8-0', tool_calls=[{'name': 'multiply', 'args': {'b': 789, 'a': 456}, 'id': 'e8760268-66b2-4295-9413-edd38469bfd9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 375, 'output_tokens': 20, 'total_tokens': 395, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='359784', name='multiply', id='f4cdf481-fd60-4eaf-8783-2c14641c6a4d',

In [11]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt="당신은 유능하고 친절한 수학 선생님입니다. 사칙 연산 요청 시 계산 우선 순위별로 단계를 나누고 계획을 세웁니다. 순서대로 Tool을 이용해 계산한 후 최종 답변을 생성합니다.",
)

In [12]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123 + 456 * 789 / 12은 뭔가요?"}]}
)

In [13]:
result

{'messages': [HumanMessage(content='123 + 456 * 789 / 12은 뭔가요?', additional_kwargs={}, response_metadata={}, id='e2f5bd4a-2d49-418d-8ce8-3a2ae766f1c9'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요! 수학 선생님입니다. 질문해주신 식을 계산하기 위해 수학의 연산 우선순위(곱셈과 나눗셈을 덧셈보다 먼저 계산)에 따라 단계를 나누어 계획을 세워보겠습니다.\n\n### **계산 계획**\n1.  **첫 번째 단계:** 곱셈을 먼저 수행합니다. ($456 \\times 789$)\n2.  **두 번째 단계:** 그 결과값을 12로 나눕니다. ($결과1 \\div 12$)\n3.  **세 번째 단계:** 마지막으로 123을 더합니다. ($123 + 결과2$)\n\n이 순서대로 차근차근 계산해 보겠습니다.\n\n---\n\n### **단계별 계산**\n\n**1단계: 456 * 789**'}], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 456, "b": 789}'}, '__gemini_function_call_thought_signatures__': {'91ceab8d-6867-4721-92f4-7313f0d7720b': 'EjQKMgEMOdbHEAYfB3q/vxlPpIW/BVBBgjpFQ3wgR4g7+jFIcChf8anEBaMvNQalrgdE2c09'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e0813-684d-7260-a1e1-f2900ac26a02-0', tool_c